# Parameter Golf: Semantic Embedding Init A/B Test

Tests whether seeding token embeddings with PPMI-SVD co-occurrence vectors
improves BPB vs random init. Zero runtime overhead — just a better starting point.

**Plan:**
1. Clone repo + download data (4 shards)
2. Baseline run (random init), ~3 min
3. Patch in PPMI-SVD embedding init, ~3 min
4. Compare BPB

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Setup

In [ ]:
!git clone https://github.com/openai/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub numpy

In [ ]:
!python data/cached_challenge_fineweb.py --train-shards 4 --variant sp1024

## 2. Baseline run (random init)

In [ ]:
!MAX_WALLCLOCK_SECONDS=180 ITERATIONS=10000 TRAIN_BATCH_TOKENS=131072 \
 VAL_LOSS_EVERY=200 TRAIN_LOG_EVERY=50 WARMUP_STEPS=10 \
 WARMDOWN_ITERS=200 RUN_ID=baseline python train_gpt.py

## 3. Patch in PPMI-SVD embedding init

Adds a `build_ppmi_svd_vectors()` function and mixes the resulting vectors
into `tok_emb.weight` before training. Zero per-step overhead — the model
just starts with semantically-informed embeddings instead of random ones.

In [ ]:
src = open("train_gpt.py", "r").read()

# --- 1. Add build_ppmi_svd_vectors function before GPT class ---

ppmi_func = '''

def build_ppmi_svd_vectors(
    shard_pattern: str, vocab_size: int, model_dim: int, window: int = 5, max_shards: int = 4,
) -> Tensor | None:
    shard_files = sorted(glob.glob(shard_pattern))[:max_shards]
    if not shard_files:
        return None
    V = vocab_size
    cooccur = np.zeros((V, V), dtype=np.float64)
    for shard_path in shard_files:
        tokens = load_data_shard(Path(shard_path)).numpy().astype(np.int64)
        n = len(tokens)
        for offset in range(1, window + 1):
            left = tokens[:n - offset]
            right = tokens[offset:]
            flat = left * V + right
            cooccur += np.bincount(flat, minlength=V * V).reshape(V, V)
            flat_rev = right * V + left
            cooccur += np.bincount(flat_rev, minlength=V * V).reshape(V, V)
    total = cooccur.sum()
    if total == 0:
        return None
    row_sums = cooccur.sum(axis=1, keepdims=True)
    col_sums = cooccur.sum(axis=0, keepdims=True)
    denom = row_sums * col_sums
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.where(
            (cooccur > 0) & (denom > 0),
            np.log(cooccur * total / denom),
            0.0,
        )
    ppmi = np.maximum(pmi, 0.0).astype(np.float32)
    rank = min(model_dim, V - 1)
    U, S, _ = torch.svd_lowrank(torch.from_numpy(ppmi), q=rank)
    vectors = (U * S.sqrt().unsqueeze(0)).numpy()
    if vectors.shape[1] < model_dim:
        vectors = np.concatenate(
            [vectors, np.zeros((V, model_dim - vectors.shape[1]), dtype=np.float32)], axis=1,
        )
    std = vectors.std()
    if std > 1e-6:
        vectors = vectors / std
    return torch.from_numpy(vectors)


'''

assert 'class GPT(nn.Module):' in src, "Can't find GPT class"
src = src.replace(
    'class GPT(nn.Module):',
    ppmi_func + 'class GPT(nn.Module):',
)

# --- 2. Add SEM_EMBED_ALPHA to Hyperparameters ---
src = src.replace(
    '    grad_clip_norm = float(os.environ.get("GRAD_CLIP_NORM", 0.0))',
    '    grad_clip_norm = float(os.environ.get("GRAD_CLIP_NORM", 0.0))\n'
    '    sem_embed_alpha = float(os.environ.get("SEM_EMBED_ALPHA", 0.005))',
)

# --- 3. Add embedding init before data loader ---
src = src.replace(
    '    train_loader = DistributedTokenLoader(args.train_files, rank, world_size, device)\n\n    def zero_grad_all',
    '    if args.sem_embed_alpha > 0:\n'
    '        import time as _time\n'
    '        _ts = _time.perf_counter()\n'
    '        _svd = build_ppmi_svd_vectors(args.train_files, args.vocab_size, args.model_dim, window=5, max_shards=4)\n'
    '        if _svd is not None:\n'
    '            with torch.no_grad():\n'
    '                base_model.tok_emb.weight.add_(_svd.to(dtype=base_model.tok_emb.weight.dtype, device=device) * args.sem_embed_alpha)\n'
    '            log0(f"sem_embed_init:alpha={args.sem_embed_alpha} built in {1000*(_time.perf_counter()-_ts):.0f}ms")\n'
    '\n'
    '    train_loader = DistributedTokenLoader(args.train_files, rank, world_size, device)\n'
    '\n'
    '    def zero_grad_all',
)

# Verify
assert 'build_ppmi_svd_vectors' in src, "Function not inserted"
assert 'sem_embed_alpha' in src, "Hyperparameter not added"

with open("train_gpt.py", "w") as f:
    f.write(src)

import ast
ast.parse(src)
print("Patch applied and syntax OK")

## 4. Semantic embedding init run

In [ ]:
!SEM_EMBED_ALPHA=0.005 MAX_WALLCLOCK_SECONDS=180 ITERATIONS=10000 \
 TRAIN_BATCH_TOKENS=131072 VAL_LOSS_EVERY=200 TRAIN_LOG_EVERY=50 \
 WARMUP_STEPS=10 WARMDOWN_ITERS=200 RUN_ID=sem_init python train_gpt.py

## 5. Results

Fill in both `val_bpb` values from the `final_int8_zlib_roundtrip_exact` lines above.

In [ ]:
baseline_bpb = None  # e.g. 1.4130
sem_init_bpb = None  # e.g. 1.4050

if baseline_bpb is not None and sem_init_bpb is not None:
    delta = sem_init_bpb - baseline_bpb
    direction = "WORSE" if delta > 0 else "BETTER" if delta < 0 else "SAME"
    print(f"Baseline BPB:        {baseline_bpb}")
    print(f"Semantic Init BPB:   {sem_init_bpb}")
    print(f"Delta: {delta:+.6f} ({direction})")
    print()
    if delta < -0.001:
        print("Semantic init is helping. Try tuning SEM_EMBED_ALPHA on EC2.")
    elif delta < 0.001:
        print("Within noise. Try different alpha values (0.001, 0.01, 0.02).")
    else:
        print("Semantic init is hurting. Try lower alpha or skip this approach.")
else:
    print("Fill in both BPB values above and re-run this cell.")

## 6. Artifact size check

In [ ]:
import os
for f in ["final_model.int8.ptz"]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"{f}: {size_mb:.2f} MB")

code_bytes = len(open("train_gpt.py").read().encode())
print(f"train_gpt.py: {code_bytes / 1024:.1f} KB")

if os.path.exists("final_model.int8.ptz"):
    total = os.path.getsize("final_model.int8.ptz") + code_bytes
    print(f"Total artifact: {total / (1024*1024):.2f} MB (limit: 16 MB)")